In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP03 - TP Services
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Did the unit use third party intermediaries or subcontractors during the assessment 
   period to perform marketing, business development, sales, consulting, or brokerage?
   
   LOGIC SUMMARY (Updated with Flattening & Bridging Architecture):
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if they meet either condition:
      - KPI_Number = '8.5' AND Value = '100'
      - primary_product_serv matches one of the 14 predefined marketing/printing categories.
   3. EXCEPTION HANDLING: Uses a 'Placeholder Swap' ([COMMA]) to protect known unit 
      names that contain commas (e.g., 'CAD PB - RESL...') from being improperly split.
   4. FLATTEN LOBs: Columns K and L can contain comma-separated lists. Uses LATERAL 
      VIEW explode(split(...)) to expand these strings into individual evaluation rows.
   5. RESTORE LOBs: Swaps '[COMMA]' back to a real comma after the explosion to match 
      the mapping table perfectly.
   6. TPRM STRING MAPPING & NAME BRIDGE: Maps expanded LOBs to AU IDs by checking if 
      the 'TPRM_Units' string exists inside the parsed LOB strings using SAFE LIKE 
      syntax (replacing the old RLIKE \b logic). Then bridges the Mapping Table's 
      String Name back to the Master List's Numeric BusinessID.
   7. FINAL OUTPUT: If a mapped AU has 1 or more flagged engagements, outputs 'Yes', 
      otherwise safely defaults to 'No'. Maintains strict 6-column structure.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- Grab valid TPRM Mapping Strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Flagged_Engagements AS (
    -- 2. Filter the TP data based on the KPI rules OR the specific service categories
    SELECT DISTINCT
        EngagementNumber,
        
        -- 3. EXCEPTION DICTIONARY: Protect commas in known LOBs using '[COMMA]'
        REPLACE(owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using
        
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND (
          -- Condition 1: KPI 8.5 equals 100
          (TRIM(KPI_Number) = '8.5' AND TRIM(Value) = '100')
          
          OR 
          
          -- Condition 2: Matches target Marketing/Printing services (Col F)
          TRIM(primary_product_serv) IN (
              'Marketing media services',
              'Marketing and distribution',
              'Direct marketing print service',
              'Print advertising',
              'Printing',
              'Management and Business Professionals and Administrative Services',
              'Marketing analysis',
              'Publicity and marketing support services',
              'Marketing communications agencies',
              'Advertising agency services',
              'Business forms or questionnaires',
              'Specialized warehousing and storage',
              'Stationery or business form printing',
              'Published Products'
          )
      )
),

Flattened_LOBs AS (
    -- 4 & 5. FLATTEN & RESTORE RULE: Split by commas, explode, then restore the protected commas
    SELECT 
        EngagementNumber, 
        -- RESTORE: Put the actual commas back so it matches the Mapping Table!
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 6. Map engagements, bridge to Master ID, and count matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    
    -- FIXED JOIN: Maps TP engagements to AU IDs (Safe LIKE replaces RLIKE)
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
        
    -- Name Bridge Join: Bridges Mapping Table Name to Master List Numeric ID
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 7. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP03' AS QuestionID,               
    
    -- Yes/No Logic based on the match count
    CASE 
        WHEN COALESCE(e.Match_Count, 0) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response,
    
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP03 - Service Category Traceability with Comma Exceptions
   
   PURPOSE: Verifies the KPI/Service flag logic, shows how the Exception Dictionary 
   safely protected comma-containing LOBs during the flattening process, and displays 
   the Master AU lookup status to troubleshoot mapping drops.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

Flattened AS (
    SELECT 
        p.EngagementNumber,
        p.KPI_Number,
        p.Value,
        p.primary_product_serv,
        
        -- The original untouched strings
        p.owning_lob AS Full_Col_K,
        p.lob_using AS Full_Col_L,
        
        -- Exception Dictionary applied
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using,
        
        TRIM(exploded_lob) AS Raw_Exploded_Chunk,
        
        -- The restored chunk
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Restored_Expanded_LOB
        
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    LATERAL VIEW explode(split(concat_ws(',', 
        REPLACE(owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'), 
        REPLACE(lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing')
    ), ',')) AS exploded_lob
    WHERE p.EngagementNumber IS NOT NULL
      AND (
          (TRIM(p.KPI_Number) = '8.5' AND TRIM(p.Value) = '100') OR 
          TRIM(p.primary_product_serv) IN (
              'Marketing media services', 'Marketing and distribution', 'Direct marketing print service',
              'Print advertising', 'Printing', 'Management and Business Professionals and Administrative Services',
              'Marketing analysis', 'Publicity and marketing support services', 'Marketing communications agencies',
              'Advertising agency services', 'Business forms or questionnaires', 'Specialized warehousing and storage',
              'Stationery or business form printing', 'Published Products'
          )
      )
      AND exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- Service Flag Traceability
    f.KPI_Number,
    f.Value,
    f.primary_product_serv AS Col_F_Service,
    
    -- LOB Flattening Traceability
    f.Full_Col_K,
    f.Full_Col_L,
    f.Restored_Expanded_LOB,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
-- Safe Mapping Join using the RESTORED string
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Restored_Expanded_LOB) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
ORDER BY f.EngagementNumber;